# COMP 469 Lab 02 — Solving Problems by Searching

**Length:** 3 hours  
**Topic:** AIMA Chapter 3, Search Algorithms  
**Environment:** This lab is self-contained and uses only the Python standard library.

## What you will build

You will implement and compare search agents for the Romania route-finding problem.

By the end of the lab, you should be able to:

- Formulate a search problem using states, actions, transition model, goal test, and path cost.
- Explain the difference between a state-space graph and a search tree.
- Implement BFS, DFS, uniform-cost search, greedy best-first search, and A*.
- Compare algorithms using completeness, optimality, node expansion, memory/frontier size, and runtime.
- Explain why a heuristic can make search faster and why greedy search can be non-optimal.

## Deliverable

Submit this notebook with:

- All TODOs completed.
- Checkpoints 1–5 answered.
- Experiment output included.
- A final reflection of about one page.

## Lab pacing

| Time | Activity |
|---|---|
| 0:00–0:20 | Setup and problem formulation |
| 0:20–0:40 | Core classes: `Problem`, `Node`, `GraphProblem` |
| 0:40–1:10 | Breadth-first search and depth-first search |
| 1:10–1:35 | Uniform-cost search |
| 1:35–2:10 | Greedy best-first search and A* |
| 2:10–2:35 | 8-puzzle heuristic functions |
| 2:35–2:55 | Controlled experiment and comparison |
| 2:55–3:00 | Final reflection and submit |

## Section 1 — Concept review

A search problem has:

- **Initial state:** where the agent starts.
- **Actions:** what the agent can do in a state.
- **Transition model:** what state results from an action.
- **Goal test:** whether the current state satisfies the objective.
- **Path cost:** total cost of the action sequence.

In this lab, each city is treated as an atomic state. The algorithms do not know what a city contains internally; they only know which states connect to which other states and at what cost.

### Checkpoint 1 — Problem formulation

Answer in this markdown cell.

1. For the Romania route problem, what is the **initial state**?
2. What is the **goal state**?
3. What are the **actions** from `Arad`?
4. What does the transition model return for `RESULT(Arad, Sibiu)`?
5. What is the action cost from `Arad` to `Sibiu`?
6. Why is this an example of an **atomic representation**?

In [ ]:
from __future__ import annotations

from collections import deque
from dataclasses import dataclass
import heapq
import itertools
import math
import time
from typing import Any, Callable, Dict, Iterable, List, Optional, Tuple


class Problem:
    """Abstract search problem.

    A problem has:
    - initial state
    - goal state or states
    - actions(state)
    - result(state, action)
    - path_cost(cost_so_far, state1, action, state2)
    - optional heuristic h(node)
    """

    def __init__(self, initial: Any, goal: Optional[Any] = None):
        self.initial = initial
        self.goal = goal

    def actions(self, state: Any) -> Iterable[Any]:
        raise NotImplementedError

    def result(self, state: Any, action: Any) -> Any:
        raise NotImplementedError

    def goal_test(self, state: Any) -> bool:
        if isinstance(self.goal, (list, tuple, set)):
            return state in self.goal
        return state == self.goal

    def path_cost(self, cost_so_far: float, state1: Any, action: Any, state2: Any) -> float:
        return cost_so_far + 1

    def h(self, node: "Node") -> float:
        return 0


@dataclass
class Node:
    """A search-tree node."""

    state: Any
    parent: Optional["Node"] = None
    action: Optional[Any] = None
    path_cost: float = 0
    depth: int = 0

    def __post_init__(self):
        if self.parent is not None:
            self.depth = self.parent.depth + 1

    def __lt__(self, other: "Node") -> bool:
        return self.path_cost < other.path_cost

    def child_node(self, problem: Problem, action: Any) -> "Node":
        next_state = problem.result(self.state, action)
        next_cost = problem.path_cost(self.path_cost, self.state, action, next_state)
        return Node(next_state, self, action, next_cost)

    def expand(self, problem: Problem) -> List["Node"]:
        return [self.child_node(problem, action) for action in problem.actions(self.state)]

    def path(self) -> List["Node"]:
        node = self
        result = []
        while node is not None:
            result.append(node)
            node = node.parent
        return list(reversed(result))

    def solution(self) -> List[Any]:
        # TODO 1:
        # Return the list of actions from the root node to this node.
        # Hint: self.path() returns nodes from root to current node.
        # Pseudocode:
        #   get the full path of nodes
        #   skip the root node because its action is None
        #   collect each remaining node.action in order
        #   return the collected actions
        raise NotImplementedError("TODO 1: implement Node.solution()")

    def path_states(self) -> List[Any]:
        # TODO 2:
        # Return the list of states from the root node to this node.
        # Pseudocode:
        #   get the full path of nodes
        #   collect each node.state in order
        #   return the collected states
        raise NotImplementedError("TODO 2: implement Node.path_states()")


class PriorityQueue:
    """Priority queue used by best-first, UCS, Greedy, and A* search."""

    def __init__(self, f: Callable[[Node], float]):
        self.f = f
        self.heap: List[Tuple[float, int, Node]] = []
        self.counter = itertools.count()

    def append(self, node: Node) -> None:
        heapq.heappush(self.heap, (self.f(node), next(self.counter), node))

    def pop(self) -> Node:
        return heapq.heappop(self.heap)[2]

    def __len__(self) -> int:
        return len(self.heap)


class Graph:
    """Weighted graph using an adjacency dictionary."""

    def __init__(self, graph_dict: Optional[Dict[Any, Dict[Any, float]]] = None, directed: bool = True):
        self.graph_dict: Dict[Any, Dict[Any, float]] = graph_dict or {}
        self.directed = directed
        if not directed:
            self.make_undirected()

    def make_undirected(self) -> None:
        for a in list(self.graph_dict.keys()):
            for b, distance in list(self.graph_dict[a].items()):
                self.connect(b, a, distance)

    def connect(self, a: Any, b: Any, distance: float = 1) -> None:
        self.graph_dict.setdefault(a, {})[b] = distance
        if not self.directed:
            self.graph_dict.setdefault(b, {})[a] = distance

    def get(self, a: Any, b: Optional[Any] = None):
        links = self.graph_dict.setdefault(a, {})
        if b is None:
            return links
        return links.get(b, math.inf)

    def nodes(self) -> List[Any]:
        result = set(self.graph_dict.keys())
        for neighbors in self.graph_dict.values():
            result.update(neighbors.keys())
        return sorted(result)


class GraphProblem(Problem):
    """Route-finding problem on a weighted graph."""

    def __init__(self, initial: Any, goal: Any, graph: Graph, heuristic: Optional[Dict[Any, float]] = None):
        super().__init__(initial, goal)
        self.graph = graph
        self.heuristic = heuristic or {}

    def actions(self, state: Any) -> Iterable[Any]:
        # TODO 3:
        # Return the neighboring cities available from this state.
        # Pseudocode:
        #   look up this state in the graph
        #   return the neighbor city names from that lookup
        raise NotImplementedError("TODO 3: implement GraphProblem.actions()")

    def result(self, state: Any, action: Any) -> Any:
        # TODO 4:
        # For this route problem, the action is the destination city.
        # Pseudocode:
        #   ignore state for this map representation
        #   return the destination named by action
        raise NotImplementedError("TODO 4: implement GraphProblem.result()")

    def path_cost(self, cost_so_far: float, state1: Any, action: Any, state2: Any) -> float:
        # TODO 5:
        # Add the road distance from state1 to state2.
        # Pseudocode:
        #   find the edge distance between state1 and state2
        #   add that distance to cost_so_far
        #   return the new total cost
        raise NotImplementedError("TODO 5: implement GraphProblem.path_cost()")

    def h(self, node: Node) -> float:
        return self.heuristic.get(node.state, 0)


romania_map = Graph(
    {
        "Arad": {"Zerind": 75, "Sibiu": 140, "Timisoara": 118},
        "Bucharest": {"Urziceni": 85, "Pitesti": 101, "Giurgiu": 90, "Fagaras": 211},
        "Craiova": {"Drobeta": 120, "Rimnicu Vilcea": 146, "Pitesti": 138},
        "Drobeta": {"Mehadia": 75},
        "Eforie": {"Hirsova": 86},
        "Fagaras": {"Sibiu": 99},
        "Hirsova": {"Urziceni": 98},
        "Iasi": {"Vaslui": 92, "Neamt": 87},
        "Lugoj": {"Timisoara": 111, "Mehadia": 70},
        "Oradea": {"Zerind": 71, "Sibiu": 151},
        "Pitesti": {"Rimnicu Vilcea": 97},
        "Rimnicu Vilcea": {"Sibiu": 80},
        "Urziceni": {"Vaslui": 142},
    },
    directed=False,
)

romania_sld_to_bucharest = {
    "Arad": 366,
    "Bucharest": 0,
    "Craiova": 160,
    "Drobeta": 242,
    "Eforie": 161,
    "Fagaras": 176,
    "Giurgiu": 77,
    "Hirsova": 151,
    "Iasi": 226,
    "Lugoj": 244,
    "Mehadia": 241,
    "Neamt": 234,
    "Oradea": 380,
    "Pitesti": 100,
    "Rimnicu Vilcea": 193,
    "Sibiu": 253,
    "Timisoara": 329,
    "Urziceni": 80,
    "Vaslui": 199,
    "Zerind": 374,
}


@dataclass
class SearchReport:
    algorithm: str
    node: Optional[Node]
    expanded: int
    generated: int
    max_frontier: int
    time_ms: float


def breadth_first_graph_search(problem: Problem, return_report: bool = False):
    start = time.perf_counter()
    node = Node(problem.initial)
    expanded = 0
    generated = 1
    max_frontier = 1

    if problem.goal_test(node.state):
        report = SearchReport("BFS", node, expanded, generated, max_frontier, (time.perf_counter() - start) * 1000)
        return report if return_report else node

    # TODO 6:
    # Create a FIFO frontier using deque.
    # Pseudocode:
    #   create a deque containing the initial node
    #   this makes the oldest frontier item come out first
    frontier = None

    reached = {problem.initial}

    while frontier:
        # TODO 7:
        # Remove the next node using FIFO behavior.
        # deque.popleft() is the key difference between BFS and DFS here.
        # Pseudocode:
        #   remove and save the leftmost frontier node
        #   that node is the next shallowest candidate to expand
        node = None

        expanded += 1

        for child in node.expand(problem):
            generated += 1
            if child.state not in reached:
                if problem.goal_test(child.state):
                    report = SearchReport("BFS", child, expanded, generated, max_frontier, (time.perf_counter() - start) * 1000)
                    return report if return_report else child

                # TODO 8:
                # Mark child.state as reached.
                # Pseudocode:
                #   record the child's state in the reached set
                #   do this before adding the child to the frontier
                pass

                # TODO 9:
                # Add child to the BFS frontier.
                # Pseudocode:
                #   append the child to the right side of the deque
                #   it will wait behind nodes already in the frontier
                pass

        max_frontier = max(max_frontier, len(frontier))

    report = SearchReport("BFS", None, expanded, generated, max_frontier, (time.perf_counter() - start) * 1000)
    return report if return_report else None


def depth_first_graph_search(problem: Problem, limit: int = 50, return_report: bool = False):
    start = time.perf_counter()
    frontier = [Node(problem.initial)]
    reached = set()
    expanded = 0
    generated = 1
    max_frontier = 1

    while frontier:
        # TODO 10:
        # Remove the next node using LIFO stack behavior.
        # Pseudocode:
        #   remove and save the most recently added frontier node
        #   that node is the deepest available candidate to expand
        node = None

        if problem.goal_test(node.state):
            report = SearchReport("DFS", node, expanded, generated, max_frontier, (time.perf_counter() - start) * 1000)
            return report if return_report else node

        if node.state not in reached and node.depth < limit:
            reached.add(node.state)
            expanded += 1
            children = list(reversed(node.expand(problem)))
            generated += len(children)
            frontier.extend(children)
            max_frontier = max(max_frontier, len(frontier))

    report = SearchReport("DFS", None, expanded, generated, max_frontier, (time.perf_counter() - start) * 1000)
    return report if return_report else None


def best_first_graph_search(problem: Problem, f: Callable[[Node], float], name: str = "Best-first", return_report: bool = False):
    start = time.perf_counter()
    node = Node(problem.initial)
    frontier = PriorityQueue(f)
    frontier.append(node)

    reached: Dict[Any, Node] = {node.state: node}
    expanded = 0
    generated = 1
    max_frontier = 1

    while frontier:
        node = frontier.pop()

        if node.path_cost > reached[node.state].path_cost:
            continue

        if problem.goal_test(node.state):
            report = SearchReport(name, node, expanded, generated, max_frontier, (time.perf_counter() - start) * 1000)
            return report if return_report else node

        expanded += 1

        for child in node.expand(problem):
            generated += 1
            if child.state not in reached or child.path_cost < reached[child.state].path_cost:
                reached[child.state] = child
                frontier.append(child)

        max_frontier = max(max_frontier, len(frontier))

    report = SearchReport(name, None, expanded, generated, max_frontier, (time.perf_counter() - start) * 1000)
    return report if return_report else None


def uniform_cost_search(problem: Problem, return_report: bool = False):
    # TODO 11:
    # Uniform-cost search uses f(n) = g(n), where g(n) is path_cost.
    # Pseudocode:
    #   define a priority score that returns a node's path_cost
    #   call best_first_graph_search with that score function
    #   pass along the algorithm name and return_report flag
    raise NotImplementedError("TODO 11: implement uniform_cost_search()")


def greedy_best_first_search(problem: Problem, return_report: bool = False):
    # TODO 12:
    # Greedy best-first search uses f(n) = h(n).
    # Pseudocode:
    #   define a priority score that returns the heuristic estimate
    #   call best_first_graph_search with that score function
    #   pass along the algorithm name and return_report flag
    raise NotImplementedError("TODO 12: implement greedy_best_first_search()")


def astar_search(problem: Problem, return_report: bool = False):
    # TODO 13:
    # A* uses f(n) = g(n) + h(n).
    # Pseudocode:
    #   define a priority score using path_cost plus heuristic estimate
    #   call best_first_graph_search with that score function
    #   pass along the algorithm name and return_report flag
    raise NotImplementedError("TODO 13: implement astar_search()")


def depth_limited_search(problem: Problem, limit: int = 50):
    def recursive_dls(node: Node, limit_remaining: int):
        if problem.goal_test(node.state):
            return node
        if limit_remaining == 0:
            return "cutoff"

        cutoff_occurred = False
        for child in node.expand(problem):
            if child.state in node.path_states():
                continue
            result = recursive_dls(child, limit_remaining - 1)
            if result == "cutoff":
                cutoff_occurred = True
            elif result is not None:
                return result

        return "cutoff" if cutoff_occurred else None

    return recursive_dls(Node(problem.initial), limit)


def iterative_deepening_search(problem: Problem, max_depth: int = 50) -> Optional[Node]:
    for depth in range(max_depth + 1):
        result = depth_limited_search(problem, depth)

        # TODO 14:
        # If result is not "cutoff", return it.
        # Otherwise continue with a deeper limit.
        # Pseudocode:
        #   run depth-limited search at the current depth
        #   if the result is a solution or failure, stop and return it
        #   if the result is cutoff, try the next depth value
        pass

    return None


def recursive_best_first_search(problem: Problem, h: Optional[Callable[[Node], float]] = None) -> Optional[Node]:
    """Provided as an extension-level algorithm. You do not need to modify this for the core lab."""
    heuristic = h or problem.h

    def rbfs(node: Node, f_limit: float) -> Tuple[Optional[Node], float]:
        if problem.goal_test(node.state):
            return node, 0

        successors = []
        current_path_states = set(node.path_states())

        for child in node.expand(problem):
            if child.state not in current_path_states:
                child.f = max(child.path_cost + heuristic(child), getattr(node, "f", 0))
                successors.append(child)

        if not successors:
            return None, math.inf

        while True:
            successors.sort(key=lambda n: n.f)
            best = successors[0]

            if best.f > f_limit:
                return None, best.f

            alternative = successors[1].f if len(successors) > 1 else math.inf
            result, best.f = rbfs(best, min(f_limit, alternative))

            if result is not None:
                return result, best.f

    start = Node(problem.initial)
    start.f = heuristic(start)
    result, _ = rbfs(start, math.inf)
    return result


class SimpleProblemSolvingAgent:
    """Simple problem-solving agent.

    The agent formulates a problem, searches for a sequence of actions, then executes one action per call.
    """

    def __init__(
        self,
        initial_state: Any,
        goal: Any,
        problem_factory: Callable[[Any, Any], Problem],
        search_algorithm: Callable[[Problem], Optional[Node]] = astar_search,
    ):
        self.state = initial_state
        self.goal = goal
        self.problem_factory = problem_factory
        self.search_algorithm = search_algorithm
        self.action_sequence: List[Any] = []

    def update_state(self, percept: Any) -> Any:
        self.state = percept
        return self.state

    def formulate_goal(self, state: Any) -> Any:
        return self.goal

    def formulate_problem(self, state: Any, goal: Any) -> Problem:
        return self.problem_factory(state, goal)

    def __call__(self, percept: Any) -> Optional[Any]:
        state = self.update_state(percept)

        if not self.action_sequence:
            goal = self.formulate_goal(state)
            problem = self.formulate_problem(state, goal)
            solution_node = self.search_algorithm(problem)
            self.action_sequence = solution_node.solution() if solution_node else []

        if self.action_sequence:
            return self.action_sequence.pop(0)

        return None


# 8-puzzle support for heuristic practice.
GOAL_8 = (0, 1, 2, 3, 4, 5, 6, 7, 8)
START_8 = (7, 2, 4, 5, 0, 6, 8, 3, 1)


def misplaced_tiles(state: Tuple[int, ...], goal: Tuple[int, ...] = GOAL_8) -> int:
    # TODO 15:
    # Count tiles that are not in their goal position.
    # Do not count the blank tile 0.
    # Pseudocode:
    #   start a counter at 0
    #   compare each tile with the tile at the same goal position
    #   when the tile is not blank and is misplaced, increment the counter
    #   return the counter
    raise NotImplementedError("TODO 15: implement misplaced_tiles()")


def manhattan_distance(state: Tuple[int, ...], goal: Tuple[int, ...] = GOAL_8) -> int:
    total = 0
    for tile in range(1, 9):
        current_index = state.index(tile)
        goal_index = goal.index(tile)
        current_row, current_col = divmod(current_index, 3)
        goal_row, goal_col = divmod(goal_index, 3)

        # TODO 16:
        # Add the vertical distance plus horizontal distance for this tile.
        # Pseudocode:
        #   compute absolute row difference
        #   compute absolute column difference
        #   add both distances to total
        raise NotImplementedError("TODO 16: implement manhattan_distance()")

    return total


def path_summary(node: Optional[Node]) -> str:
    if node is None:
        return "No solution"
    return f"{' -> '.join(node.path_states())} | cost={node.path_cost} | depth={node.depth}"


def report_table(reports: List[SearchReport]) -> None:
    header = f"{'Algorithm':<12} {'Cost':>6} {'Depth':>6} {'Expanded':>9} {'Generated':>10} {'MaxFrontier':>12} {'Time(ms)':>9}  Path"
    print(header)
    print("-" * len(header))
    for r in reports:
        if r.node is None:
            cost = depth = "-"
            path = "No solution"
        else:
            cost = int(r.node.path_cost)
            depth = r.node.depth
            path = " -> ".join(r.node.path_states())
        print(f"{r.algorithm:<12} {str(cost):>6} {str(depth):>6} {r.expanded:>9} {r.generated:>10} {r.max_frontier:>12} {r.time_ms:>9.3f}  {path}")


def text_bar(label: str, value: float, scale: float = 1.0, width: int = 50) -> str:
    count = int(value / scale)
    count = max(0, min(width, count))
    return f"{label:<12} | {'#' * count} {value}"

## Section 2 — Smoke test the problem classes

Run this after completing TODOs 1–5.

In [ ]:
romania_problem = GraphProblem("Arad", "Bucharest", romania_map, romania_sld_to_bucharest)

print("Initial:", romania_problem.initial)
print("Goal:", romania_problem.goal)
print("Actions from Arad:", list(romania_problem.actions("Arad")))
print("Result of going to Sibiu:", romania_problem.result("Arad", "Sibiu"))
print("Cost Arad -> Sibiu:", romania_problem.path_cost(0, "Arad", "Sibiu", "Sibiu"))
print("Goal test Bucharest:", romania_problem.goal_test("Bucharest"))

## Section 3 — Breadth-first search

BFS expands the shallowest nodes first. It uses a FIFO queue.

Complete TODOs 6–9, then run the cell below.

In [ ]:
bfs_report = breadth_first_graph_search(romania_problem, return_report=True)
report_table([bfs_report])

### Checkpoint 2 — BFS reasoning

Answer in this markdown cell.

1. What path did BFS find?
2. What was the path cost?
3. Is BFS guaranteed to find the cheapest route in this map? Why or why not?
4. What would happen if you accidentally used `pop()` instead of `popleft()`?

## Section 4 — Depth-first search

DFS expands the deepest available node first. It uses a LIFO stack.

Complete TODO 10, then run the cell below.

In [ ]:
dfs_report = depth_first_graph_search(romania_problem, return_report=True)
report_table([dfs_report])

### Checkpoint 3 — BFS vs. DFS

Answer in this markdown cell.

1. Did DFS find the same path as BFS?
2. Did DFS expand more or fewer nodes?
3. Why is DFS usually memory-efficient?
4. Why is DFS not guaranteed to be optimal?

## Section 5 — Uniform-cost search

Uniform-cost search expands the node with the lowest path cost so far, `g(n)`.

Complete TODO 11, then run the cell below.

In [ ]:
ucs_report = uniform_cost_search(romania_problem, return_report=True)
report_table([ucs_report])

### Checkpoint 4 — Uniform-cost search

Answer in this markdown cell.

1. What path did UCS find?
2. What was the path cost?
3. Why does UCS wait until a goal node is popped from the frontier instead of returning immediately when the goal is first generated?
4. How is UCS different from BFS when action costs are not equal?

## Section 6 — Greedy best-first search and A*

Greedy best-first search uses only the heuristic:

`f(n) = h(n)`

A* uses both the path cost so far and the heuristic estimate:

`f(n) = g(n) + h(n)`

Complete TODOs 12–13, then run the cell below.

In [ ]:
greedy_report = greedy_best_first_search(romania_problem, return_report=True)
astar_report = astar_search(romania_problem, return_report=True)

report_table([greedy_report, astar_report])

### Checkpoint 5 — Greedy vs. A*

Answer in this markdown cell.

1. Did greedy best-first search find the cheapest path?
2. Did A* find the same path as UCS?
3. Why can greedy search be fast but non-optimal?
4. What does A* add to greedy search?

## Section 7 — Depth-limited and iterative deepening search

Complete TODO 14, then run the cell below.

In [ ]:
dls_result = depth_limited_search(romania_problem, limit=3)
ids_result = iterative_deepening_search(romania_problem, max_depth=10)

print("DLS limit=3:", path_summary(dls_result if dls_result != "cutoff" else None))
print("IDS:", path_summary(ids_result))

## Section 8 — A simple problem-solving agent

The agent formulates a problem, searches for a solution, then returns one action at a time.

In [ ]:
def romania_problem_factory(state, goal):
    return GraphProblem(state, goal, romania_map, romania_sld_to_bucharest)

agent = SimpleProblemSolvingAgent(
    initial_state="Arad",
    goal="Bucharest",
    problem_factory=romania_problem_factory,
    search_algorithm=astar_search,
)

current_city = "Arad"
for step in range(5):
    action = agent(current_city)
    print(f"Step {step + 1}: at {current_city}, action = {action}")
    if action is None:
        break
    current_city = action

## Section 9 — 8-puzzle heuristic functions

The 8-puzzle is a standard search problem. We will not solve the full puzzle here; instead, you will implement two heuristic functions.

Complete TODOs 15–16.

In [ ]:
print("Start state:", START_8)
print("Goal state: ", GOAL_8)
print("Misplaced tiles:", misplaced_tiles(START_8))
print("Manhattan distance:", manhattan_distance(START_8))

### Checkpoint 6 — Heuristics

Answer in this markdown cell.

1. Which heuristic value is larger for the given 8-puzzle start state: misplaced tiles or Manhattan distance?
2. Why is Manhattan distance usually more informative?
3. Why must an admissible heuristic avoid overestimating the true cost?

## Section 10 — Controlled experiment

Compare BFS, DFS, UCS, Greedy, and A* on several route-finding pairs.

Fill in the short written analysis after running the experiment.

In [ ]:
test_pairs = [
    ("Arad", "Bucharest"),
    ("Sibiu", "Bucharest"),
    ("Timisoara", "Bucharest"),
    ("Neamt", "Craiova"),
    ("Oradea", "Eforie"),
]

all_reports = []

for start_city, goal_city in test_pairs:
    print(f"\n=== {start_city} -> {goal_city} ===")

    # For this lab, the provided heuristic table is only straight-line distance to Bucharest.
    # A* and Greedy are most meaningful when the goal is Bucharest.
    heuristic = romania_sld_to_bucharest if goal_city == "Bucharest" else {}

    problem = GraphProblem(start_city, goal_city, romania_map, heuristic)

    reports = [
        breadth_first_graph_search(problem, return_report=True),
        depth_first_graph_search(problem, return_report=True),
        uniform_cost_search(problem, return_report=True),
    ]

    if goal_city == "Bucharest":
        reports.append(greedy_best_first_search(problem, return_report=True))
        reports.append(astar_search(problem, return_report=True))

    report_table(reports)
    all_reports.extend(reports)

### Text chart: nodes expanded

This avoids external plotting dependencies. Run the cell below after the experiment.

In [ ]:
for report in all_reports:
    print(text_bar(report.algorithm, report.expanded, scale=1))

## Final reflection

Write about one page answering the following:

1. Which algorithm performed best overall for finding low-cost paths?
2. Which algorithm expanded the fewest nodes?
3. Why is "fewest nodes expanded" not always the same as "best path"?
4. What did this lab show about the tradeoff between optimality, time, and memory?
5. How does A* connect the ideas of path cost and heuristic estimates?

Use at least three specific numbers from your own experiment output.

## Optional extension challenge

Choose one if you finish early.

**Option A:** Implement weighted A* with `f(n) = g(n) + w*h(n)` for `w = 1.5` and `w = 2.0`.

**Option B:** Add a new route pair and compare the algorithms.

**Option C:** Modify the Romania graph by closing one road. How does the best route change?